# Task 2.3 – Interesting Finds

Three reproducible discoveries from the Oscar Award dataset, queried via **PonyORM** against `oscar.db`.

**Dataset:** 11,110 nominations · 97 ceremonies (1927–2024) · 6,893 persons · 5,233 films  
**Kernel:** `oscar-py311` (Python 3.11 — required for PonyORM generator expressions)

In [9]:
# ── Imports & ORM setup ───────────────────────────────────────────────────────
import os
from collections import defaultdict

import pandas as pd
from pony.orm import (
    Database, PrimaryKey, Required, Optional, Set, composite_key,
    db_session, select, count, desc,
)

BASE_DIR = os.path.dirname(os.path.abspath("task2_oscar_findings.ipynb"))
DB_PATH  = os.path.join(BASE_DIR, "oscar.db")

db = Database()

class Ceremony(db.Entity):
    number      = PrimaryKey(int)
    year        = Required(int)
    nominations = Set("Nomination")

class Category(db.Entity):
    name        = Required(str, unique=True)
    canon_name  = Required(str)
    nominations = Set("Nomination")

class Person(db.Entity):
    name        = Required(str, unique=True)
    nominations = Set("Nomination")

class Film(db.Entity):
    title       = Required(str)
    year        = Required(int)
    nominations = Set("Nomination")
    composite_key(title, year)

class Nomination(db.Entity):
    ceremony = Required(Ceremony)
    category = Required(Category)
    person   = Optional(Person)
    film     = Optional(Film)
    winner   = Required(bool)

db.bind(provider="sqlite", filename=DB_PATH, create_db=False)
db.generate_mapping()
print("DB connected:", DB_PATH)

DB connected: c:\Users\user\OneDrive\PycharmProjects\Big Data scraping\task2\oscar.db


---
## Finding 1 — Angela Lansbury holds the record for the longest gap between first nomination and first win: 69 years

Which person waited the longest from their first nomination to their first win?  
**Approach:** fetch every `(person_name, ceremony_year, winner)` triple via a PonyORM
generator expression, then compute `min(win_years) - min(nom_years)` per person in Python.

In [10]:
# Step 1: fetch all nomination records that have a named person
with db_session:
    all_noms = select(
        (n.person.name, n.ceremony.year, n.winner, n.category.canon_name)
        for n in Nomination
        if n.person is not None
    )[:]

# Step 2: group by person and compute the gap in Python
person_data = defaultdict(lambda: {"nom_years": [], "win_years": [], "win_cats": []})
for name, year, winner, canon_name in all_noms:
    person_data[name]["nom_years"].append(year)
    if winner:
        person_data[name]["win_years"].append(year)
        person_data[name]["win_cats"].append((year, canon_name))

gaps = []
for name, data in person_data.items():
    if data["win_years"]:          # only people who eventually won
        gap = min(data["win_years"]) - min(data["nom_years"])
        if gap > 0:
            first_win_year = min(data["win_years"])
            first_win_cat = next(
                cat for yr, cat in sorted(data["win_cats"]) if yr == first_win_year
            )
            gaps.append({
                "Person":          name,
                "Gap (years)":     gap,
                "First Nom":       min(data["nom_years"]),
                "First Win":       first_win_year,
                "First Win Category": first_win_cat,
            })

gaps.sort(key=lambda x: -x["Gap (years)"])
df_gaps = pd.DataFrame(gaps[:15])
print("Top 15 longest gaps between first nomination and first win:")
display(df_gaps)

# Step 3: verify how many of those 15 first wins are honorary vs competitive
HONORARY_CATS = {"HONORARY AWARD", "JEAN HERSHOLT HUMANITARIAN AWARD", "SPECIAL AWARD",
                 "IRVING G. THALBERG MEMORIAL AWARD", "GORDON E. SAWYER AWARD"}
honorary_count = sum(1 for row in gaps[:15] if row["First Win Category"] in HONORARY_CATS)
competitive_count = 15 - honorary_count
print(f"\nOf the top 15 longest-gap individuals:")
print(f"  Honorary/lifetime awards as first win : {honorary_count}")
print(f"  Competitive awards as first win       : {competitive_count}")

Top 15 longest gaps between first nomination and first win:


,Person,Gap (years),First Nom,First Win,First Win Category
0,Angela Lansbury,69,1945,2014,HONORARY AWARD
1,Brazil,62,1963,2025,INTERNATIONAL FEATURE FILM
2,Mexico,58,1961,2019,INTERNATIONAL FEATURE FILM
3,King Vidor,51,1928,1979,HONORARY AWARD
4,Poland,51,1964,2015,INTERNATIONAL FEATURE FILM
5,Debbie Reynolds,51,1965,2016,JEAN HERSHOLT HUMANITARIAN AWARD
6,Lalo Schifrin,51,1968,2019,HONORARY AWARD
7,Piero Tosi,50,1964,2014,HONORARY AWARD
8,Ralph Bellamy,49,1938,1987,HONORARY AWARD
9,George Stevens Jr.,49,1964,2013,HONORARY AWARD



Of the top 15 longest-gap individuals:
  Honorary/lifetime awards as first win : 11
  Competitive awards as first win       : 4


In [11]:
# Zoom in on Angela Lansbury's full nomination history
with db_session:
    lansbury = Person.get(name="Angela Lansbury")
    noms = sorted(lansbury.nominations, key=lambda n: n.ceremony.year)
    records = [
        {
            "Year":     n.ceremony.year,
            "Category": n.category.canon_name,
            "Film":     n.film.title if n.film else "—",
            "Win":      "Yes" if n.winner else "",
        }
        for n in noms
    ]

print("Angela Lansbury — complete Oscar record:")
display(pd.DataFrame(records))

Angela Lansbury — complete Oscar record:


,Year,Category,Film,Win
0,1945,ACTRESS IN A SUPPORTING ROLE,Gaslight,
1,1946,ACTRESS IN A SUPPORTING ROLE,The Picture of Dorian Gray,
2,1963,ACTRESS IN A SUPPORTING ROLE,The Manchurian Candidate,
3,2014,HONORARY AWARD,—,Yes


**Why it's interesting:**  
69 years is longer than most film careers. Lansbury was nominated before she turned 20 and
only won after she turned 88. The table above also reveals a structural pattern: of the top 15
longest-gap individuals in the dataset, **11 of the eventual first "wins" are Honorary Awards or
Jean Hersholt Humanitarian Awards** — suggesting the Academy systematically uses honorary
recognition to compensate for careers it repeatedly overlooked in the competitive race.
The remaining 4 competitive wins in the top 15 all belong to countries in the
International Feature Film category (Brazil, Mexico, Poland, Japan), not individuals.

---
## Finding 2 — Titanic (1997) is the only film nominated in 14 distinct canonical categories

Which film reached across the most distinct departments of filmmaking?  
**Approach:** count `distinct canon_name` values per film using PonyORM's `count(..., distinct=True)`.

In [12]:
# Films ranked by number of distinct canonical categories nominated in
with db_session:
    results = select(
        (n.film.title, n.film.year, count(n.category.canon_name, distinct=True))
        for n in Nomination
        if n.film is not None
    ).order_by(lambda t, y, cnt: desc(cnt))[:15]

    rows = []
    for title, year, distinct_cats in results:
        total_noms = count(
            n for n in Nomination
            if n.film is not None and n.film.title == title and n.film.year == year
        )
        wins = count(
            n for n in Nomination
            if n.film is not None and n.film.title == title
            and n.film.year == year and n.winner
        )
        rows.append({
            "Film":            title,
            "Year":            year,
            "Distinct Categories": distinct_cats,
            "Total Nominations":   total_noms,
            "Wins":            wins,
            "Win Rate":        f"{wins / total_noms:.0%}" if total_noms else "—",
        })

df_films = pd.DataFrame(rows)
print("Top 15 films by distinct canonical categories nominated in:")
display(df_films)

Top 15 films by distinct canonical categories nominated in:


,Film,Year,Distinct Categories,Total Nominations,Wins,Win Rate
0,Titanic,1997,14,14,11,79%
1,Forrest Gump,1994,13,13,6,46%
2,Gone with the Wind,1939,13,14,9,64%
3,La La Land,2016,13,14,6,43%
4,Mary Poppins,1964,13,13,5,38%
5,Oppenheimer,2023,13,13,7,54%
6,Shakespeare in Love,1998,13,13,7,54%
7,The Curious Case of Benjamin Button,2008,13,13,3,23%
8,The Lord of the Rings: The Fellowship of the Ring,2001,13,13,4,31%
9,The Shape of Water,2017,13,13,4,31%


In [13]:
# Show every category Titanic was nominated in
with db_session:
    titanic = Film.get(title="Titanic", year=1997)
    categories = sorted({n.category.canon_name for n in titanic.nominations})

print(f"Titanic (1997) — {len(categories)} distinct canonical categories:")
for cat in categories:
    print(f"  • {cat}")

Titanic (1997) — 14 distinct canonical categories:
  • ACTRESS IN A LEADING ROLE
  • ACTRESS IN A SUPPORTING ROLE
  • ART DIRECTION
  • BEST PICTURE
  • CINEMATOGRAPHY
  • COSTUME DESIGN
  • DIRECTING
  • FILM EDITING
  • MAKEUP AND HAIRSTYLING
  • MUSIC (Original Score)
  • MUSIC (Original Song)
  • SOUND EDITING
  • SOUND MIXING
  • VISUAL EFFECTS


**Why it's interesting:**  
Seven films sit at 13 distinct categories; *Titanic* stands alone at 14. Comparing it to
*Ben-Hur* (1959) — which also won 11 Oscars — shows the difference:
*Ben-Hur* achieved a 92% win rate from 12 nominations while *Titanic* won 11 of 14 (79%).
The breadth of *Titanic*'s nominations reflects how completely the Academy judged it the
dominant achievement across every craft discipline simultaneously — a feat unmatched since.

---
## Finding 3 — Kenneth Branagh is the only person nominated competitively in all three primary filmmaking roles: Acting, Directing, and Writing

Who has been recognised by the Academy as simultaneously an actor, a director, and a writer?  
**Approach:** for each person, classify their nominations into craft disciplines and find those who span all three primary creative roles — using only competitive nominations (no honorary awards).

In [ ]:
ACTING_CATS   = {"ACTOR IN A LEADING ROLE", "ACTRESS IN A LEADING ROLE",
                 "ACTOR IN A SUPPORTING ROLE", "ACTRESS IN A SUPPORTING ROLE",
                 "ACTOR", "ACTRESS"}
DIRECTING_CATS = {"DIRECTING"}
WRITING_CATS  = {"WRITING (Adapted Screenplay)", "WRITING (Original Screenplay)",
                 "WRITING (Story and Screenplay)", "WRITING"}
HONORARY_CATS = {"HONORARY AWARD", "JEAN HERSHOLT HUMANITARIAN AWARD", "SPECIAL AWARD",
                 "IRVING G. THALBERG MEMORIAL AWARD", "GORDON E. SAWYER AWARD",
                 "BEST PICTURE"}   # Best Picture credits the producer, not a personal craft role

with db_session:
    persons_list = select(p for p in Person)[:]
    triple = []   # nominated in all three: Acting + Directing + Writing
    double = []   # nominated in exactly two of the three
    for p in persons_list:
        # Only competitive craft nominations
        craft_cats = {n.category.canon_name for n in p.nominations
                      if n.category.canon_name not in HONORARY_CATS}
        has_acting  = bool(craft_cats & ACTING_CATS)
        has_dir     = bool(craft_cats & DIRECTING_CATS)
        has_writing = bool(craft_cats & WRITING_CATS)
        total_noms  = len(p.nominations)
        wins        = sum(1 for n in p.nominations if n.winner)
        if has_acting and has_dir and has_writing:
            triple.append({"Person": p.name, "Nominations": total_noms, "Wins": wins,
                           "Disciplines": "Acting + Directing + Writing"})
        elif has_acting and (has_dir or has_writing):
            roles = ["Acting"]
            if has_dir:     roles.append("Directing")
            if has_writing: roles.append("Writing")
            double.append({"Person": p.name, "Nominations": total_noms, "Wins": wins,
                           "Disciplines": " + ".join(roles)})

triple.sort(key=lambda x: -x["Nominations"])
double.sort(key=lambda x: -x["Nominations"])

print("People with competitive nominations in Acting + Directing + Writing:")
display(pd.DataFrame(triple))

print("\nPeople with competitive nominations in exactly two of the three disciplines (top 10):")
display(pd.DataFrame(double[:10]))

In [ ]:
# Zoom in on Kenneth Branagh — show his full nomination history to confirm the breadth
with db_session:
    branagh = Person.get(name="Kenneth Branagh")
    noms = sorted(branagh.nominations, key=lambda n: n.ceremony.year)
    records = [
        {
            "Year":       n.ceremony.year,
            "Film":       n.film.title if n.film else "—",
            "Category":   n.category.canon_name,
            "Win":        "Yes" if n.winner else "",
        }
        for n in noms
    ]

print("Kenneth Branagh — complete Oscar record:")
display(pd.DataFrame(records))

**Why it's interesting:**  
Out of ~6,893 nominees across 97 years, only **4 individuals** have competitive craft
nominations in all three primary filmmaking disciplines (Acting, Directing, and Writing):
Woody Allen, John Huston, Kenneth Branagh, and John Cassavetes.

What makes Branagh's case stand out within this group is the *spread*: his 6 nominations
cover every sub-role — Actor in a Leading Role, Actor in a Supporting Role, Directing (twice),
Writing (Adapted Screenplay), and Writing (Original Screenplay). He was nominated for
Directing and Leading Actor for the *same film* (*Henry V*, 1990), returned with a Writing
nomination for *Hamlet* (1997), and then 25 years later earned both Directing and Writing
nominations for *Belfast* (2022), winning the latter. Every one of his nominations is a
competitive craft recognition with no honorary awards inflating the count.

Woody Allen has the highest total (21 nominations) but his acting credit is a single entry —
his dominance is in Writing (13) and Directing (7), making him a specialist in two disciplines
rather than a generalist across three.

---
## Summary

| Finding | Key figure | Discovered via |
|---|---|---|
| Longest gap (nomination → win) | Angela Lansbury: 69 years | Fetch all `(person, year, winner, category)` triples; compute gap in Python; 11/15 longest-gap wins are honorary |
| Film in most distinct categories | Titanic (1997): 14 categories | `count(canon_name, distinct=True)` per film |
| Only person nominated in all 3 crafts | 4 individuals total; Branagh's 6 nominations are all competitive with no honorary inflation | Filter craft nominations only; check intersection with Acting, Directing, Writing sets |

All queries use only **PonyORM generator expressions** — no raw SQL. The `oscar.db` file
built in Task 2.1 is the only data source.